# 무릎·발목이 보이는 세 장에 수행자 전신 박스 그리기
**모든 셀 실행** 후 2번 화면에서 세 탭을 차례로 여세요. a08 29번은 학습, b02 36번과 c04 78번은 검증입니다. 노란 십자는 무릎 관절선, 노란 마름모는 발목 복사뼈입니다. a08은 오른쪽 두 관절 판독, b02·c04는 기존 1차 직접 판독을 사용합니다.

### 1. 세 장의 원본과 분할 확인

In [ ]:
import ipywidgets as widgets
import pandas as pd
from IPython.display import display

from wlcv.annotation.performer_box import PerformerBox
from wlcv.annotation.pose_box_editor import performer_box_editor
from wlcv.annotation.pose_points import pose_point_path
from wlcv.config import ANNOTATIONS, ROOT, SPLIT, VIDEOS

videos = pd.read_csv(ROOT / "data/metadata/videos.csv").set_index("video_id", verify_integrity=True)
roles = pd.read_csv(SPLIT).set_index("video_id", verify_integrity=True)
# 영상 ID, 프레임 번호(0부터), 분할, 가까운 쪽, 기존 관절 판독의 상위 폴더
scenes = [
    ("a08_sn_f_01", 29, "train", "right", ANNOTATIONS / "a08_right_knee_ankle"),
    ("b02_sn_h_02", 36, "validation", "right", ANNOTATIONS),
    ("c04_sn_f_01", 78, "validation", "left", ANNOTATIONS),
]
folder = ANNOTATIONS / "pose_box_review"
split_names = {"train": "학습", "validation": "검증"}
side_names = {"right": "오른쪽", "left": "왼쪽"}

### 2. 박스 그리기와 저장
한 장면에서 **수행자의 머리부터 발끝까지** 감싸도록 사진 위의 한 모서리에서 대각선 반대편까지 드래그하세요. 원판이나 뒤에 있는 사람은 박스의 기준에 포함하지 않습니다. 주황색 박스를 확인하고 **박스 저장**을 누르면 초록색으로 바뀝니다. 고치려면 다시 드래그하고 저장하세요. **저장본 다시 열기**를 누르면 마지막으로 저장된 박스가 표시됩니다.

In [ ]:
panels = []
titles = []
for video, frame, split, side, joint_folder in scenes:
    info = videos.loc[video]
    role = roles.loc[video]
    if (role.role, role.person_id, info.near_side) != (split, info.person_id, side):
        raise ValueError(f"{video}의 분할·선수·방향을 확인하세요.")
    source = pose_point_path(joint_folder, video, frame, pass_number=1)
    box = PerformerBox(folder / split, video, frame)
    editor = performer_box_editor(VIDEOS / info.filename, frame, source=source,
                                  person=info.person_id, split=split, side=side, box=box)
    panels.append(editor)
    titles.append(f"{info.person_id} {frame}번 · {split_names[split]} · {side_names[side]}")

tabs = widgets.Tab(children=panels)
for index, title in enumerate(titles):
    tabs.set_title(index, title)
display(tabs)

### 3. 저장한 박스 확인
판독 후 이 셀을 다시 실행하면 완료 장수와 원본 픽셀 좌표를 볼 수 있습니다. 박스 CSV는 `data/annotations/pose_box_review/` 아래의 `train/person_boxes/` 또는 `validation/person_boxes/`에 저장됩니다.

In [ ]:
rows = []
for video, frame, split, side, _ in scenes:
    saved = PerformerBox(folder / split, video, frame).saved
    rows.append({"영상": video, "프레임": frame, "분할": split_names[split],
                 "박스 저장": saved is not None, "좌표 (xmin, ymin, xmax, ymax)": saved})
report = pd.DataFrame(rows)
print(f"{report['박스 저장'].sum()}/3장 저장")
display(report)